# IT2011 – AI/ML Solution for Physical Fitness Classification

**Group:** 2026-Y2-S1-KU-16

**Member:** Dias L.I.K.  
**Admission (IT) Number:** IT25101521

**Assigned Preprocessing Technique (Step 1 of 6):** Data Cleaning — Standardizing Mixed Data Types

---
This notebook contains the full shared group pipeline (imports, data loading, EDA, preprocessing, modeling, evaluation, and export) so it runs end-to-end. Within the **Preprocessing** cell below, this member's specific assigned step is marked with ⭐, and the steps owned by other teammates are labelled for reference.


In [ ]:
# System & File Handling
import os
import joblib
import warnings
from google.colab import files

# Data Handling & Analytics
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Model Selection
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.exceptions import InconsistentVersionWarning

# ML Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')
print("✅ Libraries imported successfully!")

In [ ]:
# 1. Upload CSV dataset
print("Upload 'fitness_dataset.csv' or 'fitness_dataset_2.csv':")
uploaded = files.upload()

filename = list(uploaded.keys())[0]

# 2. Load dataset into DataFrame
df = pd.read_csv(filename)

# 3. Overview of dataset characteristics
print("\n--- Dataset Summary ---")
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}")
print("\nColumn Data Types & Missing Values:")
print(df.info())

print("\nFirst 5 Rows:")
display(df.head())

print("\nTarget Variable Distribution ('is_fit'):")
print(df['is_fit'].value_counts())

In [ ]:
# Set visual style
sns.set_theme(style="whitegrid")
plt.figure(figsize=(16, 10))

# Plot 1: Target distribution
plt.subplot(2, 2, 1)
sns.countplot(data=df, x='is_fit', palette='Set2')
plt.title("Fitness Target Distribution (0 = Not Fit, 1 = Fit)")
plt.xlabel("Fitness Status")
plt.ylabel("Count")

# Plot 2: Heart Rate vs Blood Pressure by Fitness Class
plt.subplot(2, 2, 2)
sns.scatterplot(data=df, x='heart_rate', y='blood_pressure', hue='is_fit', palette='Set1', alpha=0.7)
plt.title("Heart Rate vs Blood Pressure by Fitness Class")

# Plot 3: Activity Index Distribution
plt.subplot(2, 2, 3)
sns.boxplot(data=df, x='is_fit', y='activity_index', palette='Set3')
plt.title("Activity Index across Fitness Classes")

# Plot 4: Correlation Heatmap
plt.subplot(2, 2, 4)
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap='coolwarm', cbar=False)
plt.title("Feature Correlation Matrix")

plt.tight_layout()
plt.show()

In [ ]:
# ⭐ MY ASSIGNED TASK — Step 1: Data Cleaning — Standardizing Mixed Data Types
# Owner: Dias L.I.K. (IT25101521)
# 1. Standardize string inconsistencies in categorical feature 'smokes'
df['smokes'] = df['smokes'].astype(str).str.lower().replace({'0': 'no', '1': 'yes'})

# (Step 2 — Feature/Target Separation (X / y split) — assigned to Medagama M.S., IT25101527)
# 2. Separate features (X) and target variable (y)
X = df.drop(columns=['is_fit'])
y = df['is_fit']

# (Step 3 — Stratified Train-Test Split — assigned to Marasinghe M.M.B.I., IT25101538)
# 3. Perform Stratified Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature categorizations
num_cols = ['age', 'height_cm', 'weight_kg', 'heart_rate', 'blood_pressure',
            'sleep_hours', 'nutrition_quality', 'activity_index']
cat_cols = ['smokes', 'gender']

# (Step 4 — Missing Value Imputation (Median Imputation) — assigned to Ranasinghe R.M.M.K., IT25101541)
# 4. Missing value median imputation
imputer = SimpleImputer(strategy='median')
X_train[num_cols] = imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = imputer.transform(X_test[num_cols])

# (Step 5 — Categorical Encoding (Label Encoding) — assigned to Ukwaththa U.K.A.A.N., IT25101545)
# 5. Categorical Encoding (Label Encoding)
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])
    label_encoders[col] = le

# (Step 6 — Feature Scaling (StandardScaler) — assigned to Rathnayaka R.M.B.G.T.A.B.R., IT25101550)
# 6. Feature Scaling (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(" preprocessing completed successfully!")
print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

In [ ]:
print("🔍 Hyperparameter tuning with GridSearchCV in progress...\n")

# 1. Logistic Regression Tuning
param_grid_lr = {'C': [0.01, 0.1, 1, 10], 'solver': ['liblinear', 'lbfgs']}
grid_lr = GridSearchCV(LogisticRegression(random_state=42), param_grid_lr, cv=5, scoring='accuracy')
grid_lr.fit(X_train_scaled, y_train)

# 2. Decision Tree Tuning
param_grid_dt = {'max_depth': [3, 5, 7, 10], 'min_samples_split': [2, 5, 10]}
grid_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid_dt, cv=5, scoring='accuracy')
grid_dt.fit(X_train_scaled, y_train)

# 3. Random Forest Tuning
param_grid_rf = {'n_estimators': [50, 100, 150], 'max_depth': [5, 10, None], 'min_samples_split': [2, 5]}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=5, scoring='accuracy')
grid_rf.fit(X_train_scaled, y_train)

best_models = {
    'Logistic Regression': grid_lr.best_estimator_,
    'Decision Tree': grid_dt.best_estimator_,
    'Random Forest': grid_rf.best_estimator_
}

print("--- Best Hyperparameters Found ---")
print("Logistic Regression:", grid_lr.best_params_)
print("Decision Tree:", grid_dt.best_params_)
print("Random Forest:", grid_rf.best_params_)

In [ ]:
# 1. Evaluate models on Test Set
results = []
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, model) in enumerate(best_models.items()):
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    # 5-Fold Cross-Validation on Training Data
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=kf, scoring='accuracy')

    results.append({
        'Model': name,
        'Test Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1-Score': round(f1_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_proba), 4),
        '5-Fold CV Mean Acc': f"{cv_scores.mean():.4f} (±{cv_scores.std():.4f})"
    })

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)
    axes[idx].set_title(f"{name}\nConfusion Matrix")
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("Actual")

plt.tight_layout()
plt.show()

# Display performance comparison table
comparison_df = pd.DataFrame(results)
print("\n--- Model Performance Comparison ---")
display(comparison_df)

In [ ]:
# Compute median dictionary explicitly to avoid scikit-learn class version mismatches
medians_dict = X_train[num_cols].median().to_dict()

# Bundle model and transformers
pipeline_bundle = {
    'model': best_models['Logistic Regression'],
    'medians': medians_dict,  # Stores raw numeric medians instead of SimpleImputer object
    'label_encoders': label_encoders,
    'scaler': scaler,
    'feature_names': list(X.columns),
    'num_cols': num_cols,
    'cat_cols': cat_cols
}

# Save artifact
output_filename = 'fitness_model_pipeline.pkl'
joblib.dump(pipeline_bundle, output_filename)

print(f"✅ Saved robust pipeline successfully as '{output_filename}'!")
files.download(output_filename)

In [ ]:
def predict_single_sample(sample_data: dict, pipeline_path='fitness_model_pipeline.pkl'):
    """
    Takes a raw input dictionary and returns fitness classification with confidence score.
    """
    # Load pipeline
    bundle = joblib.load(pipeline_path)
    model = bundle['model']
    medians = bundle['medians'] # Changed from imputer
    encoders = bundle['label_encoders']
    scaler = bundle['scaler']
    features = bundle['feature_names']
    num_cols = bundle['num_cols']
    cat_cols = bundle['cat_cols']

    # Convert dictionary to DataFrame
    df_sample = pd.DataFrame([sample_data])

    # 1. Categorical cleaning
    if 'smokes' in df_sample.columns:
        df_sample['smokes'] = df_sample['smokes'].astype(str).str.lower().replace({'0': 'no', '1': 'yes'})

    # 2. Median Imputation
    # Use the medians dictionary to fill missing values
    for col in num_cols:
        if col in df_sample.columns and df_sample[col].isnull().any():
            df_sample[col].fillna(medians[col], inplace=True)

    # 3. Label Encoding
    for col in cat_cols:
        # Handle cases where a category might not have been seen during training
        # using errors='ignore' will set unknown categories to NaN, which might need further handling
        # For this specific pipeline, it's assumed categories will match trained encoders.
        df_sample[col] = encoders[col].transform(df_sample[col])

    # 4. Reorder features
    df_sample = df_sample[features]

    # 5. Standard Scaling
    sample_scaled = scaler.transform(df_sample)

    # 6. Prediction & Probabilities
    pred = int(model.predict(sample_scaled)[0])
    probabilities = model.predict_proba(sample_scaled)[0]

    return {
        "prediction": pred,
        "classification": "Fit (1)" if pred == 1 else "Not Fit (0)",
        "confidence_score": round(float(probabilities[pred]), 4)
    }

# --- Example Demonstration ---
sample_person = {
    'age': 28,
    'height_cm': 178,
    'weight_kg': 72,
    'heart_rate': 62.0,
    'blood_pressure': 118.0,
    'sleep_hours': 7.5,
    'nutrition_quality': 8.5,
    'activity_index': 4.2,
    'smokes': 'no',
    'gender': 'M'
}

result = predict_single_sample(sample_person)
print("\n--- Single Sample Prediction Result ---")
print(result)